# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [14]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [15]:
print(OPENAI_API_KEY is not None)


True


In [16]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [17]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [18]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

Verily, it didst cross the road to showeth the possum that 'twas possible.


In [19]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you doing today?


In [20]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to your name or personal information. If you'd like to share your name with me, I'll be happy to remember it during our conversation.


In [21]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa!


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [22]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [23]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'9b110e04-2f47-44f5-8d09-b1cd7aef8f36': {'version…

In [24]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)
print(messages)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total price": 23.45
}
[{'role': 'system', 'content': "\nYou are OrderBot, an automated service to collect orders for a pizza restaurant. You first greet the customer, then collects the order, and then asks if it's a pickup or delivery. You wait to collect the entire order, then summarize it and check for a final time if the customer wants to add anything else. If it's a delivery, you ask for an address. Finally you collect the payment.Make sure to clarify all options, extras and sizes to uniquely identify the item from the menu.You respond in a short, very conversational friendly style. The menu includes pepperoni pizza  12.95, 10.00, 7.00 cheese pizza   10.95, 9.25, 6.50 eggplant pizza   11.95, 9.75, 6.75 

## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Exercise Answer

I tried 3 different chatbot prompt versions. I kept the same idea from class: the `system` message gives the bot its job, then the `user` messages test if it follows that job.

### Version 1 - Pizza OrderBot, But More Strict

This version is close to the class example, but I made it stricter about sizes, prices, and not inventing menu items.

In [25]:
messages_v1 = [
    {
        "role": "system",
        "content": """
        You are OrderBot for a pizza restaurant.
        Your job is to collect a full customer order.
        Be friendly and short.
        Only sell items from this menu:
        - pepperoni pizza: large 12.95, medium 10.00, small 7.00
        - cheese pizza: large 10.95, medium 9.25, small 6.50
        - eggplant pizza: large 11.95, medium 9.75, small 6.75
        - fries: large 4.50, small 3.50
        - greek salad: 7.25
        - toppings: extra cheese 2.00, mushrooms 1.50, sausage 3.00, peppers 1.00
        - drinks: coke, sprite, bottled water
        Always ask for missing size, toppings, pickup/delivery, and address if delivery.
        Do not invent menu items.
        """
    },
    {"role": "user", "content": "Hi, I want a pepperoni pizza and a drink."},
]

response_v1 = get_completion_from_messages(messages_v1, temperature=0)
print(response_v1)

Sure! What size pepperoni pizza would you like, and which drink would you prefer?


### Version 2 - Healthy Meal Assistant

Here I changed the bot from a pizza shop into a simple healthy meal assistant. I wanted to see if changing only the system message could completely change the chatbot behavior.

In [26]:
messages_v2 = [
    {
        "role": "system",
        "content": """
        You are a friendly healthy meal assistant.
        Help the customer choose a simple meal from this menu:
        - grilled chicken bowl: 11.50
        - tofu bowl: 10.50
        - salmon salad: 13.00
        - lentil soup: 6.50
        - side salad: 4.00
        - water: 1.50
        - green tea: 2.00
        Ask about dietary preference first: meat, fish, vegetarian, or vegan.
        If the user asks for something unhealthy or not on the menu, politely suggest the closest menu item.
        Keep the answer short.
        """
    },
    {"role": "user", "content": "I want something filling but vegetarian."},
]

response_v2 = get_completion_from_messages(messages_v2, temperature=0.3)
print(response_v2)

I recommend the tofu bowl for $10.50. It's a delicious and filling vegetarian option on our menu.


### Version 3 - Study Tutor Bot

This one is not a restaurant bot. I made a study assistant that explains concepts and then asks a checking question. This was to test how much the role changes the whole conversation.

In [27]:
messages_v3 = [
    {
        "role": "system",
        "content": """
        You are a beginner-friendly Python tutor.
        Explain things simply, with small examples.
        After explaining, ask one short question to check if the student understood.
        Do not give a long lecture.
        """
    },
    {"role": "user", "content": "Can you explain what a Python dictionary is?"},
]

response_v3 = get_completion_from_messages(messages_v3, temperature=0.2)
print(response_v3)

Sure! A Python dictionary is a collection of key-value pairs. Each key is unique and is used to access its corresponding value. Dictionaries are enclosed in curly braces {} and each key-value pair is separated by a colon (:).

For example:
```python
my_dict = {'name': 'Alice', 'age': 30, 'city': 'New York'}
```

In this example, 'name', 'age', and 'city' are keys, and 'Alice', 30, and 'New York' are their corresponding values.

Can you give an example of a key-value pair in a dictionary?


### Extra Test - A Bot That Can Hallucinate More Easily

This prompt is weaker because it tells the bot to be creative but does not clearly limit it. I expect this one to be more likely to invent details.

In [28]:
messages_bad = [
    {
        "role": "system",
        "content": """
        You are a creative restaurant assistant.
        Make the customer excited and suggest interesting food combinations.
        """
    },
    {"role": "user", "content": "What pizzas do you have and how much are they?"},
]

response_bad = get_completion_from_messages(messages_bad, temperature=0.9)
print(response_bad)

We have a variety of delicious pizzas to tantalize your taste buds! How about trying our classic Margherita pizza topped with fresh basil and creamy mozzarella, or our spicy Diablo pizza with jalapeños and pepperoni? We also have gourmet options like our Fig and Prosciutto pizza with sweet fig jam and savory prosciutto.

Our pizzas range in price from $12 to $18, depending on the toppings and size. And if you're feeling adventurous, you can create your own unique pizza combination! How about adding some pineapple and jalapeños for a sweet and spicy kick, or trying our BBQ Chicken pizza with tangy barbecue sauce and grilled chicken? The possibilities are endless, and we're excited to help you find the perfect pizza to satisfy your cravings!


## Report

In this lab I learned the whole flow of building a simple LLM chatbot, not only how to make a pizza bot. The main idea is that the OpenAI chat API does not just receive one plain question. It receives a list of messages. Each message has a role and content. The important roles are system, user, and assistant.

The system message is where we give the model its job, rules, and style. For example, in the OrderBot example, the system message tells the model that it works for a pizza restaurant, gives it the menu, and tells it to ask about pickup, delivery, address, and payment. The user messages are what the human writes. The assistant messages are the model's previous replies.

The biggest thing I learned is that the LLM does not automatically remember a conversation. We create the memory ourselves with the context list. Every time the user sends a message, the code adds it to context. Then the model gives an answer, and the code adds that answer to context too. The next time we call the API, we send the whole context again. That is why the chatbot can remember things like the pizza size or toppings from earlier in the conversation.

I also learned that prompt quality matters a lot. In my three experiments, the strict pizza bot worked best because it had clear menu items, prices, and rules. The healthy meal assistant worked too, but it was more open-ended. The Python tutor bot showed that the same chat format can be used for a totally different task. So the important lesson is not really pizza. The important lesson is how changing the system message changes the chatbot behavior.

The weaker experiment was the creative restaurant assistant. Since I did not give it a real menu or prices, it could easily invent things. That helped me understand hallucination better. If we want exact answers, we need to give the model exact context and clear limits. If the prompt is vague, the model has more room to guess.

The Panel part of the lab was the user interface. It gives us the text box, button, and chat display. But behind the interface, the real logic is: get the user input, append it to context, call the model, append the assistant answer, and display the result.

My main takeaway is: an LLM app is not just the model. It is the model plus good instructions, roles, conversation memory, an API call, and sometimes a UI. The code around the model is what makes it behave like a useful chatbot.